# 11 - Business Recommendations

## Customer Intelligence Platform

This notebook applies the rule-based recommendation engine to generate
actionable retention strategies for every customer.

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.load_data import load_features
from src.recommend import (
    generate_recommendation,
    batch_recommend,
    recommendation_summary,
    classify_risk,
    classify_clv_tier,
    RULES,
)
from src.config import TARGET

%matplotlib inline


In [ ]:
df = load_features()

# Add churn probability and CLV if available
import joblib, json
from src.config import MODELS_DIR, BEST_MODEL_FILE
from src.preprocessing import encode_features

model = joblib.load(BEST_MODEL_FILE)
with open(MODELS_DIR / "feature_names.json") as f:
    feature_names = json.load(f)

df_encoded = encode_features(df.copy())
X = df_encoded.drop(columns=[TARGET], errors="ignore")
for feat in feature_names:
    if feat not in X.columns:
        X[feat] = 0
X = X[feature_names]

scaler = joblib.load(MODELS_DIR / "scaler.joblib")
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges",
            "service_count", "revenue_per_month", "revenue_intensity",
            "tech_adoption_score", "risk_score", "contract_tenure"]
scale_cols = [c for c in num_cols if c in X.columns]
X[scale_cols] = scaler.transform(X[scale_cols])

df["churn_probability"] = model.predict_proba(X)[:, 1]
df["clv"] = df["Monthly Charges"] * df["Tenure Months"]
print(f"Data prepared: {len(df):,} customers")


## 1. Recommendation Rules

Our rule-based engine uses 9 prioritized rules:


In [ ]:
print("Recommendation Rules:")
print("=" * 60)
for rule in RULES:
    print(f"\n  Priority {rule['priority']}: {rule['name']}")
    print(f"    Impact: {rule['expected_impact']}")
    print(f"    Cost: {rule['estimated_cost']}")


## 2. Apply Recommendations


In [ ]:
df_rec = batch_recommend(df)
print(f"Recommendations generated for {len(df_rec):,} customers")


## 3. Recommendation Distribution


In [ ]:
rec_dist = df_rec["recommendation"].value_counts()
print("Recommendation Distribution:")
print(rec_dist)

fig, ax = plt.subplots(figsize=(10, 6))
rec_dist.plot(kind="barh", ax=ax, color="#3498db", edgecolor="white")
ax.set_title("Recommendation Distribution", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Customers")
plt.tight_layout()
plt.show()


## 4. Priority Analysis


In [ ]:
# High-priority actions (Priority 1-3)
high_priority = df_rec[df_rec["priority"] <= 3]
print(f"HIGH PRIORITY Actions:")
print(f"  Customers: {len(high_priority):,}")
if "Monthly Charges" in high_priority.columns:
    print(f"  Monthly Revenue at Risk: ${high_priority['Monthly Charges'].sum():,.0f}")
if "churn_probability" in high_priority.columns:
    print(f"  Average Churn Probability: {high_priority['churn_probability'].mean():.1%}")


## 5. Example Recommendations


In [ ]:
# Show examples from each recommendation type
for rec_name in df_rec["recommendation"].unique():
    subset = df_rec[df_rec["recommendation"] == rec_name]
    print(f"\n{'='*60}")
    print(f"  {rec_name} ({len(subset):,} customers)")
    print(f"{'='*60}")
    if "action" in subset.columns:
        print(f"  Action: {subset.iloc[0]['action'][:200]}")
    if "churn_probability" in subset.columns:
        print(f"  Avg Churn Prob: {subset['churn_probability'].mean():.1%}")
    if "Monthly Charges" in subset.columns:
        print(f"  Avg Monthly: ${subset['Monthly Charges'].mean():.0f}")


## Summary

### Recommendation Strategy Matrix

| Recommendation | Target Segment | Expected Impact | Investment |
|---------------|---------------|----------------|------------|
| Premium Retention | High CLV + High Risk | High | $$$ |
| Contract Migration | Month-to-Month + At Risk | High | $$ |
| Security Bundle | Fiber w/o Security | Medium | $$ |
| New Customer Onboarding | New + At Risk | Medium | $ |
| Payment Transition | Electronic Check + Risk | Medium | $ |
| Service Enrichment | Low Services + Medium Risk | Medium | $ |
| Loyalty Recognition | High CLV + Low Risk | Low | $ |
| Growth Nurture | Medium/Low CLV + Low Risk | Low | $ |
| Standard Monitoring | All Others | Low | - |

### Expected Business Impact
- Prioritized interventions for the highest-value at-risk customers
- Estimated coverage of all customer segments
- Cost-tiered strategies matching investment to expected return
